In [4]:
"""
config.py
=========
Central configuration for the Kalshi × Google Trends × S&P 500 Fear & Greed project.

Replace all placeholder values with real credentials before running.
"""

# ── Kalshi ─────────────────────────────────────────────────────────────────
# The project targets the Federal Reserve / FOMC Rate Decision series.
# These markets price the probability of the Fed *raising* rates ≥ 25bps,
# which has a direct, well-documented link to equity-market volatility.
#
# Ticker strategy (see 01_extract_kalshi.py for discovery helper):
#   Series prefix : KXFED   (Fed Funds rate cut / no-cut markets)
#   Example tickers that cover the 2024–2025 window:
#       KXFED-25JAN29    – "Will the Fed cut rates at the Jan 2025 meeting?"
#       KXFED-25MAR19    – March 2025 meeting
#       KXFED-25MAY07    – May 2025 meeting
#       KXFED-25JUN18    – June 2025 meeting
#   The extraction script pulls ALL matching tickers automatically via
#   GET /historical/markets with series_ticker="KXFED".

KALSHI_BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
KALSHI_SERIES   = "KXFED"                       # Fed rate-decision series

# Daily candlestick interval (minutes)
KALSHI_PERIOD   = 1440  # 1 day

# ── Time window ────────────────────────────────────────────────────────────
# 2024-01-01 → 2025-12-31  (2 full calendar years, ~500 trading days)
# Enough history to span multiple FOMC meetings while keeping API calls light.
START_TS = 1704067200   # 2024-01-01 00:00 UTC
END_TS   = 1767225600   # 2026-01-01 00:00 UTC

START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"

# ── Google Trends ──────────────────────────────────────────────────────────
# Three search terms chosen to proxy "information demand" around Fed / macro:
#   1. "Federal Reserve"  – direct information search about the policy actor
#   2. "interest rate"    – broader monetary-policy anxiety
#   3. "stock market crash" – explicit fear/panic signal
#
# The pytrends library returns *weekly* data for windows > 270 days.
# Strategy: pull overlapping 90-day windows and stitch/normalise to daily.
# See 02_extract_trends.py for the full stitching logic.

TRENDS_TERMS     = ["Federal Reserve", "interest rate", "stock market crash"]
TRENDS_GEO       = "US"           # United States only
TRENDS_GPROP     = ""             # web search (not news/youtube)

# ── Market data ────────────────────────────────────────────────────────────
# Downloaded via yfinance – no API key needed.
SP500_TICKER = "^GSPC"
VIX_TICKER   = "^VIX"

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "C:/Python/CSUREMM/data"
OUTPUTS_DIR = "C:/Python/CSUREMM/outputs"

In [13]:
"""
01_extract_kalshi.py
====================
Extract daily candlestick data for ALL Kalshi Fed-rate-decision markets
(series KXFED) that fall within the configured time window.

Why KXFED?
-----------
Each FOMC meeting spawns a binary market: "Will the Fed cut rates at
meeting X?".  The YES price (0–1) is the crowd's real-money probability
of a cut.  Rising probability → risk-on / equities up; falling → risk-off.
This gives us a continuous, daily macro-sentiment signal that is
fundamentally linked to S&P 500 moves.

Ticker challenge & solution
----------------------------
Kalshi does NOT provide a single aggregated "series price".  Each FOMC
meeting has its own ticker (e.g. KXFED-25MAR19).  The API is rigid:
one ticker per candlestick call.

Solution implemented here:
  1.  Discover tickers via GET /historical/markets with series_ticker filter.
  2.  Pull candlesticks for each ticker separately.
  3.  For any given calendar day, aggregate across concurrent live markets:
        - kalshi_prob_mean  : equal-weight mean of all active YES prices
        - kalshi_prob_nearest: price of the *nearest upcoming meeting*
        - kalshi_volume_total: total daily volume across all markets
        - kalshi_oi_total   : total open interest
  4.  Save raw per-ticker CSVs + aggregated panel CSV.
"""

import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone

KALSHI_BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
KALSHI_SERIES   = "KXFED"                       # Fed rate-decision series

# Daily candlestick interval (minutes)
KALSHI_PERIOD   = 1440  # 1 day

# ── Time window ────────────────────────────────────────────────────────────
# 2024-01-01 → 2025-12-31  (2 full calendar years, ~500 trading days)
# Enough history to span multiple FOMC meetings while keeping API calls light.
START_TS = 1704067200   # 2024-01-01 00:00 UTC
END_TS   = 1767225600   # 2026-01-01 00:00 UTC

START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "C:/Python/CSUREMM/data"

os.makedirs(DATA_DIR, exist_ok=True)

# ── Auth header ────────────────────────────────────────────────────────────
HEADERS = {
    "Content-Type":  "application/json",
}

# ── Helpers ────────────────────────────────────────────────────────────────

def safe_float(x):
    """Return float or NaN for None/missing values."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return float("nan")


def ts_to_date(ts: int) -> str:
    """Convert Unix timestamp (seconds) to ISO date string."""
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")


# All timestamp field names Kalshi has used across API versions
_TS_FIELDS = [
    "expiration_ts", "close_ts", "expiry_ts",
    "settlement_ts", "end_ts", "close_time",
    "expiration_time",
]
 
def _extract_settle_ts(m: dict) -> int:
    """
    Try every known timestamp field name.  Also handles ISO-8601 strings
    (e.g. "2025-01-29T18:00:00Z") by converting them to Unix seconds.
    Returns 0 if nothing is found.
    """
    for field in _TS_FIELDS:
        val = m.get(field)
        if val is None:
            continue
        if isinstance(val, (int, float)) and val > 0:
            return int(val)
        if isinstance(val, str) and val:
            try:
                from dateutil import parser as dp
                return int(dp.parse(val).timestamp())
            except Exception:
                pass
    return 0
 
 
def discover_tickers(series: str) -> list[dict]:
    """
    Paginate through GET /historical/markets filtering by series_ticker.
    Returns a list of market dicts (ticker, title, settlement date, …).
 
    Robustness fixes vs original:
      • Prints first raw market dict so unknown field names are visible.
      • Falls back to ticker-string date parsing if no timestamp field found.
      • Accepts markets whose ticker contains '24' or '25' (year digits) as a
        last-resort heuristic when ALL timestamps are 0.
    """
    url     = f"{KALSHI_BASE_URL}/historical/markets"
    all_mkt = []
    cursor  = None
 
    print(f"[discover] Fetching markets for series '{series}' …")
    while True:
        params: dict = {"series_ticker": series, "limit": 200}
        if cursor:
            params["cursor"] = cursor
 
        resp = requests.get(url, headers=HEADERS, params=params)
        resp.raise_for_status()
        body = resp.json()
 
        markets = body.get("markets", [])
        all_mkt.extend(markets)
        print(f"  … fetched {len(markets)} (total: {len(all_mkt)})")
 
        # First page only: dump raw body keys so field-name issues are obvious
        if len(all_mkt) == len(markets) and markets:
            print(f"  [debug] Top-level response keys: {list(body.keys())}")
 
        cursor = body.get("cursor")
        if not cursor or not markets:
            break
        time.sleep(0.3)
 
    if not all_mkt:
        return []
 
    # ── Debug: show field names from first market so issues are visible ────
    sample = all_mkt[0]
    print(f"\n[discover] Sample market keys : {list(sample.keys())}")
    print(f"[discover] Sample market values: {sample}\n")
 
    # ── Attempt 1: filter by timestamp fields ──────────────────────────────
    filtered = []
    ts_found = False
    for m in all_mkt:
        ts = _extract_settle_ts(m)
        if ts > 0:
            ts_found = True
            if START_TS <= ts <= END_TS:
                filtered.append(m)
 
    if ts_found and filtered:
        print(f"[discover] {len(filtered)} markets in window "
              f"{START_DATE}→{END_DATE} (via timestamp field)")
        return filtered
 
    # ── Attempt 2: parse date from ticker string ───────────────────────────
    # Kalshi tickers often encode the meeting date, e.g.:
    #   KXFED-25MAR19   → March 19 2025
    #   KXFED-24DEC18   → December 18 2024
    import re
    MONTH_MAP = {m: i+1 for i, m in enumerate(
        ["JAN","FEB","MAR","APR","MAY","JUN",
         "JUL","AUG","SEP","OCT","NOV","DEC"])}
 
    filtered_by_ticker = []
    for m in all_mkt:
        ticker = m.get("ticker", "")
        # Pattern: two-digit year + three-letter month + two-digit day
        match = re.search(r"(\d{2})([A-Z]{3})(\d{2})$", ticker)
        if match:
            yy, mon, dd = match.groups()
            year  = 2000 + int(yy)
            month = MONTH_MAP.get(mon, 0)
            if month:
                try:
                    dt = datetime(year, month, int(dd), tzinfo=timezone.utc)
                    ts = int(dt.timestamp())
                    if START_TS <= ts <= END_TS:
                        filtered_by_ticker.append(m)
                except ValueError:
                    pass
 
    if filtered_by_ticker:
        print(f"[discover] {len(filtered_by_ticker)} markets in window "
              f"{START_DATE}→{END_DATE} (via ticker date parsing)")
        return filtered_by_ticker
 
    # ── Attempt 3: widen the window (return ALL markets as last resort) ────
    print(f"[discover][WARN] Could not filter by date; "
          f"returning all {len(all_mkt)} markets. "
          f"Candlestick calls will self-filter by start_ts/end_ts.")
    return all_mkt

In [15]:
# ── Step 2 : Pull candlesticks for one ticker ──────────────────────────────
 
def fetch_candlesticks(ticker: str) -> pd.DataFrame:
    """
    Call GET /historical/markets/{ticker}/candlesticks for daily bars.
    Falls back to the live endpoint if the market is still within the
    live-data window (Kalshi automatically routes it, but we try both).
    """
    endpoints = [
        f"{KALSHI_BASE_URL}/historical/markets/{ticker}/candlesticks",
        f"{KALSHI_BASE_URL}/markets/{ticker}/candlesticks",          # live fallback
    ]
 
    params = {
        "start_ts":        START_TS,
        "end_ts":          END_TS,
        "period_interval": KALSHI_PERIOD,
    }
 
    for url in endpoints:
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if resp.status_code == 404:
                continue          # try next endpoint
            resp.raise_for_status()
            data = resp.json()
            candles = data.get("candlesticks", [])
            if not candles:
                return pd.DataFrame()
            return _parse_candles(candles, ticker)
        except requests.HTTPError as e:
            print(f"  [warn] {ticker} HTTP {e.response.status_code} at {url}")
            continue
 
    return pd.DataFrame()
 
 
def _parse_candles(candles: list, ticker: str) -> pd.DataFrame:
    """Parse raw candle dicts into a tidy DataFrame."""
    rows = []
    for c in candles:
        price   = c.get("price",   {}) or {}
        yes_ask = c.get("yes_ask", {}) or {}
        yes_bid = c.get("yes_bid", {}) or {}
 
        end_ts = c.get("end_period_ts")
        row = {
            "date":           ts_to_date(end_ts) if end_ts else None,
            "end_period_ts":  end_ts,
            "ticker":         ticker,
            # Mid-price (best estimate of consensus probability)
            "price_close":    safe_float(price.get("close")),
            "price_open":     safe_float(price.get("open")),
            "price_high":     safe_float(price.get("high")),
            "price_low":      safe_float(price.get("low")),
            "price_mean":     safe_float(price.get("mean")),
            # Bid / Ask
            "yes_bid_close":  safe_float(yes_bid.get("close")),
            "yes_ask_close":  safe_float(yes_ask.get("close")),
            # Mid of bid-ask spread as robustness check
            "yes_mid_close":  (safe_float(yes_bid.get("close")) +
                               safe_float(yes_ask.get("close"))) / 2,
            # Activity
            "volume":         safe_float(c.get("volume")),
            "open_interest":  safe_float(c.get("open_interest")),
        }
        rows.append(row)
 
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

In [16]:
# ── Step 3 : Aggregate across markets by day ───────────────────────────────
 
def aggregate_daily(all_dfs: list[pd.DataFrame],
                    market_meta: list[dict]) -> pd.DataFrame:
    """
    For each calendar day produce a single row:
      - kalshi_prob_nearest : YES price of the soonest upcoming meeting
      - kalshi_prob_mean    : equal-weight mean across all active markets
      - kalshi_volume_total : sum of volume
      - kalshi_oi_total     : sum of open interest
      - kalshi_spread_mean  : mean bid-ask spread (liquidity proxy)
      - n_markets_active    : # of markets with non-null price on that day
    """
    if not all_dfs:
        return pd.DataFrame()
 
    combined = pd.concat(all_dfs, ignore_index=True)
 
    # Build a settlement-date lookup so we can find the nearest market per day
    import re
    MONTH_MAP = {m: i+1 for i, m in enumerate(
        ["JAN","FEB","MAR","APR","MAY","JUN",
         "JUL","AUG","SEP","OCT","NOV","DEC"])}
 
    settle_map = {}
    for m in market_meta:
        t  = m.get("ticker", "")
        ts = _extract_settle_ts(m)
 
        # Fallback: parse date from ticker string if no timestamp found
        if ts == 0:
            match = re.search(r"(\d{2})([A-Z]{3})(\d{2})$", t)
            if match:
                yy, mon, dd = match.groups()
                year  = 2000 + int(yy)
                month = MONTH_MAP.get(mon, 0)
                if month:
                    try:
                        dt = datetime(year, month, int(dd), tzinfo=timezone.utc)
                        ts = int(dt.timestamp())
                    except ValueError:
                        pass
 
        if ts:
            settle_map[t] = pd.Timestamp(ts_to_date(ts))
 
    combined["settle_date"] = combined["ticker"].map(settle_map)
    combined["spread"]      = combined["yes_ask_close"] - combined["yes_bid_close"]
 
    # Group by day
    agg_rows = []
    for day, grp in combined.groupby("date"):
        # Only consider markets that haven't settled yet as of `day`
        active = grp[grp["settle_date"] >= day].dropna(subset=["price_close"])
        if active.empty:
            # Fallback: use all markets (e.g., settled-same-day)
            active = grp.dropna(subset=["price_close"])
        if active.empty:
            continue
 
        # Nearest market = smallest settle_date - day
        active = active.copy()
        active["days_to_settle"] = (active["settle_date"] - day).dt.days
        nearest_row = active.loc[active["days_to_settle"].idxmin()]
 
        agg_rows.append({
            "date":                   day,
            "kalshi_prob_nearest":    nearest_row["price_close"],
            "kalshi_prob_mean":       active["price_close"].mean(),
            "kalshi_prob_weighted":   (                         # volume-weighted
                (active["price_close"] * active["volume"]).sum()
                / active["volume"].replace(0, float("nan")).sum()
            ),
            "kalshi_volume_total":    active["volume"].sum(),
            "kalshi_oi_total":        active["open_interest"].sum(),
            "kalshi_spread_mean":     active["spread"].mean(),
            "n_markets_active":       len(active),
            "nearest_ticker":         nearest_row["ticker"],
        })
 
    agg = pd.DataFrame(agg_rows).sort_values("date").reset_index(drop=True)
    return agg

In [17]:
# ── Main ───────────────────────────────────────────────────────────────────

def main():
    # 1. Discover tickers
    market_meta = discover_tickers(KALSHI_SERIES)
    if not market_meta:
        print("[ERROR] No markets found – check API key and series ticker.")
        return
 
    # Save metadata
    meta_df = pd.DataFrame(market_meta)
    meta_df.to_csv(f"{DATA_DIR}/kalshi_market_meta.csv", index=False)
    print(f"[save] {DATA_DIR}/kalshi_market_meta.csv  ({len(meta_df)} rows)")
 
    # 2. Fetch candlesticks per ticker
    all_dfs = []
    for m in market_meta:
        ticker = m.get("ticker", "")
        if not ticker:
            continue
        print(f"  [fetch] {ticker} …", end=" ", flush=True)
        df = fetch_candlesticks(ticker)
        if df.empty:
            print("0 candles")
        else:
            print(f"{len(df)} candles")
            all_dfs.append(df)
            # Save raw per-ticker file
            df.to_csv(f"{DATA_DIR}/kalshi_{ticker}.csv", index=False)
        time.sleep(0.25)    # polite pacing
 
    if not all_dfs:
        print("[ERROR] No candlestick data retrieved.")
        return
 
    # 3. Aggregate
    agg = aggregate_daily(all_dfs, market_meta)
    agg.to_csv(f"{DATA_DIR}/kalshi_daily.csv", index=False)
    print(f"\n[save] {DATA_DIR}/kalshi_daily.csv  ({len(agg)} rows)")
    print(agg.head(10).to_string(index=False))
 
 
if __name__ == "__main__":
    main()

[discover] Fetching markets for series 'KXFED' …
  … fetched 200 (total: 200)
  [debug] Top-level response keys: ['cursor', 'markets']
  … fetched 169 (total: 369)

[discover] Sample market keys : ['can_close_early', 'close_time', 'created_time', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'floor_strike', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'strike_type', 'subtitle', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes

In [1]:
"""
02_extract_trends.py
====================
Pull Google Trends search indices for three Fed/macro-anxiety terms at
US-national daily granularity and stitch overlapping windows into a
continuous normalised series.

Terms
-----
  1. "Federal Reserve"     – information demand about the policy actor
  2. "interest rate"       – broader monetary-policy search
  3. "stock market crash"  – explicit fear / panic signal

The pytrends daily-data challenge
-----------------------------------
Google Trends returns DAILY data only for windows ≤ 270 days.
For longer windows it silently switches to WEEKLY aggregation.

Stitching strategy (overlap-anchor method):
  1.  Divide the full window into overlapping 90-day chunks
      (step = 60 days, overlap = 30 days on each side).
  2.  Pull daily data for each chunk.
  3.  In the overlap region, compute the ratio of the *new* series to the
      *anchor* series; multiply the new chunk by this ratio to align scales.
  4.  Concatenate, take the mean where overlaps remain, and re-normalise
      the full series to 0–100.
  5.  Save the final aligned daily DataFrame.

Fallback: if pytrends daily requests fail for a chunk (rate-limit or
Google server error), retry up to 3 times with exponential back-off,
then interpolate missing days from adjacent data.

Dependencies: pip install pytrends pandas
"""

import os
import time
import warnings
import pandas as pd
import numpy as np
from datetime import timedelta
from pytrends.request import TrendReq

START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"

TRENDS_TERMS     = ["Federal Reserve", "interest rate", "stock market crash"]
TRENDS_GEO       = "US"           # United States only
TRENDS_GPROP     = ""             # web search (not news/youtube)

DATA_DIR    = "C:/Python/CSUREMM/data"
OUTPUTS_DIR = "C:/Python/CSUREMM/outputs"

os.makedirs(DATA_DIR, exist_ok=True)
warnings.filterwarnings("ignore")

# ── Constants ──────────────────────────────────────────────────────────────
CHUNK_DAYS   = 90    # window per request (≤ 270 → daily resolution)
STEP_DAYS    = 60    # advance step between windows (30-day overlap on right)
MAX_RETRIES  = 3
RETRY_WAIT   = 30    # seconds (doubles each retry)


# ── pytrends session ───────────────────────────────────────────────────────
def make_pytrends() -> TrendReq:
    return TrendReq(
        hl="en-US",
        tz=0,           # UTC
        timeout=(10, 35),
        retries=2,
        backoff_factor=0.5,
    )


# ── Pull one chunk ─────────────────────────────────────────────────────────

def pull_chunk(pytrends: TrendReq, term: str,
               start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """
    Fetch daily Google Trends index for `term` between start and end.
    Returns a pd.Series indexed by date (UTC, daily), values 0–100.
    Returns empty Series on failure.
    """
    timeframe = f"{start.strftime('%Y-%m-%d')} {end.strftime('%Y-%m-%d')}"
    wait = RETRY_WAIT

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(
                kw_list=[term],
                cat=0,
                timeframe=timeframe,
                geo=TRENDS_GEO,
                gprop=TRENDS_GPROP,
            )
            df = pytrends.interest_over_time()
            if df.empty:
                return pd.Series(dtype=float)

            # Drop the `isPartial` flag column if present
            if "isPartial" in df.columns:
                df = df.drop(columns=["isPartial"])

            s = df[term].rename(None)
            s.index = pd.to_datetime(s.index).normalize()

            # If Google returned weekly data (> 270-day window or edge case):
            # resample to daily via forward-fill then flag for caller.
            if len(s) > 0 and (s.index[-1] - s.index[0]).days > 0:
                freq = pd.infer_freq(s.index)
                if freq and "W" in str(freq):
                    print(f"    [warn] weekly data returned for '{term}' "
                          f"({timeframe}); resampling to daily.")
                    s = s.resample("D").interpolate(method="linear")

            return s.astype(float)

        except Exception as e:
            print(f"    [retry {attempt}/{MAX_RETRIES}] '{term}' {timeframe}: {e}")
            time.sleep(wait)
            wait *= 2
            pytrends = make_pytrends()   # fresh session

    return pd.Series(dtype=float)



In [2]:
# ── Stitch chunks into a continuous daily series ───────────────────────────

def stitch_term(term: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """
    Build a continuous daily index for `term` over [start, end] 
    using the overlap-anchor stitching method.
    """
    # Defensive programming check to completely prevent infinite loops
    if STEP_DAYS <= 0:
        raise ValueError("STEP_DAYS must be a positive integer greater than 0.")
        
    pytrends = make_pytrends()
    chunks = []
    chunk_start = start
    
    while chunk_start <= end:
        # Use pd.Timedelta for robust timestamp date stepping
        chunk_end = min(chunk_start + pd.Timedelta(days=CHUNK_DAYS - 1), end)
        print(f"  chunk {chunk_start.date()} → {chunk_end.date()} …", end=" ", flush=True)
        
        try:
            s = pull_chunk(pytrends, term, chunk_start, chunk_end)
            if s.empty:
                print("empty (skipped)")
            else:
                print(f"{len(s)} days")
                chunks.append(s)
        except Exception as e:
            # Prevent whole pipeline crashes on a single spotty network call
            print(f"error (skipped): {str(e)}")
            
        # Shift ahead using Pandas native delta alignment
        chunk_start += pd.Timedelta(days=STEP_DAYS)
        time.sleep(5)  # Cooldown window to prevent 429 Rate Limit blocks
        
    if not chunks:
        return pd.Series(dtype=float, name=term)
        
    # ── Overlap-anchor stitching ──────────────────────────────────────────
    stitched = chunks[0].copy()
    
    for new in chunks[1:]:
        if new.empty:
            continue
            
        # Find overlapping dates
        overlap_dates = stitched.index.intersection(new.index)
        
        # Ensure we have enough intersecting points to calculate a meaningful slope/ratio
        if len(overlap_dates) >= 2:
            anchor_vals = stitched.loc[overlap_dates]
            new_vals = new.loc[overlap_dates]
            
            # Use epsilon + max logic to guard against zero-division errors if search volume dips to 0
            denom = new_vals.mean()
            if denom <= 1e-9:
                ratio = 1.0  # Fallback gracefully to standard concat if secondary window yields zero data
            else:
                ratio = anchor_vals.mean() / denom
                
            new_scaled = new * ratio
        else:
            # Fallback path if no overlap matches exist
            new_scaled = new.copy()
            
        # Merge: Concat and calculate structural means for overlap junctions
        combined = pd.concat([stitched, new_scaled], axis=0)
        stitched = combined.groupby(level=0).mean()
        
    # Re-index to enforce a strictly daily grid over the entire target horizon
    full_idx = pd.date_range(start, end, freq="D")
    stitched = stitched.reindex(full_idx)
    
    # Smooth over missing fragments using full linear interpolation
    stitched = stitched.interpolate(method="linear").ffill().bfill()
    
    # Re-normalise onto the standardized Google Trends 0–100 coordinate bounds
    mn, mx = stitched.min(), stitched.max()
    if mx > mn:
        stitched = 100 * (stitched - mn) / (mx - mn)
        
    stitched.name = term
    return stitched


In [3]:
# ── Main ───────────────────────────────────────────────────────────────────

def main():
    start = pd.Timestamp(START_DATE)
    end   = pd.Timestamp(END_DATE)

    all_series = {}
    for term in TRENDS_TERMS:
        print(f"\n[trends] Processing '{term}' …")
        s = stitch_term(term, start, end)
        all_series[term] = s
        print(f"  → {len(s)} daily observations")

    trends_df = pd.DataFrame(all_series)
    trends_df.index.name = "date"

    # Sanitise column names for use in file paths and SQL
    trends_df.columns = [
        c.lower().replace(" ", "_") for c in trends_df.columns
    ]

    out_path = f"{DATA_DIR}/google_trends_daily.csv"
    trends_df.to_csv(out_path)
    print(f"\n[save] {out_path}  shape={trends_df.shape}")
    print(trends_df.head(10).to_string())


if __name__ == "__main__":
    main()


[trends] Processing 'Federal Reserve' …
  chunk 2024-01-01 → 2024-03-30 … 90 days
  chunk 2024-03-01 → 2024-05-29 … 90 days
  chunk 2024-04-30 → 2024-07-28 … 90 days
  chunk 2024-06-29 → 2024-09-26 … 90 days
  chunk 2024-08-28 → 2024-11-25 … 90 days
  chunk 2024-10-27 → 2025-01-24 … 90 days
  chunk 2024-12-26 → 2025-03-25 … 90 days
  chunk 2025-02-24 → 2025-05-24 … 90 days
  chunk 2025-04-25 → 2025-07-23 … 90 days
  chunk 2025-06-24 → 2025-09-21 … 90 days
  chunk 2025-08-23 → 2025-11-20 … 90 days
  chunk 2025-10-22 → 2025-12-31 … 71 days
  chunk 2025-12-21 → 2025-12-31 … 11 days
  → 731 daily observations

[trends] Processing 'interest rate' …
  chunk 2024-01-01 → 2024-03-30 … 90 days
  chunk 2024-03-01 → 2024-05-29 … 90 days
  chunk 2024-04-30 → 2024-07-28 … 90 days
  chunk 2024-06-29 → 2024-09-26 … 90 days
  chunk 2024-08-28 → 2024-11-25 … 90 days
  chunk 2024-10-27 → 2025-01-24 … 90 days
  chunk 2024-12-26 → 2025-03-25 … 90 days
  chunk 2025-02-24 → 2025-05-24 … 90 days
  chunk 202

In [ ]:
"""
03_extract_market_data.py
=========================
Download daily S&P 500 (^GSPC) and CBOE VIX (^VIX) data via yfinance.

No API key required. Data includes:
  - Adjusted closing prices
  - Daily returns
  - Realised 5-day & 21-day rolling volatility (as VIX complement)
  - Forward 1-day and 5-day S&P 500 returns (dependent variables for models)

Dependencies: pip install yfinance pandas
"""

import os
import numpy as np
import pandas as pd
import yfinance as yf

START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"

# ── Market data ────────────────────────────────────────────────────────────
# Downloaded via yfinance – no API key needed.
SP500_TICKER = "^GSPC"
VIX_TICKER   = "^VIX"

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "C:/Python/CSUREMM/data"
OUTPUTS_DIR = "C:/Python/CSUREMM/outputs"

os.makedirs(DATA_DIR, exist_ok=True)


def download(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download OHLCV + Adj Close for a single ticker."""
    print(f"[yfinance] Downloading {ticker} …", end=" ", flush=True)
    # add one day buffer so we don't lose the last observation to yfinance's
    # exclusive-end convention
    df = yf.download(ticker, start=start, end=end,
                     auto_adjust=True, progress=False)
    df.index = pd.to_datetime(df.index).normalize()
    df.index.name = "date"
    print(f"{len(df)} rows")
    return df


def build_sp500_features(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich the raw S&P 500 DataFrame with return and vol features."""
    out = pd.DataFrame(index=df.index)

    price = df["Close"].squeeze()
    out["sp500_close"]       = price
    out["sp500_return_1d"]   = price.pct_change(1)
    out["sp500_return_5d"]   = price.pct_change(5)
    out["sp500_return_21d"]  = price.pct_change(21)

    # Forward returns (what we ultimately want to predict / correlate)
    out["sp500_fwd_1d"]      = price.pct_change(1).shift(-1)
    out["sp500_fwd_5d"]      = price.pct_change(5).shift(-5)

    # Realised volatility (annualised)
    log_ret = np.log(price / price.shift(1))
    out["sp500_rv5"]         = log_ret.rolling(5).std()  * np.sqrt(252)
    out["sp500_rv21"]        = log_ret.rolling(21).std() * np.sqrt(252)

    # Price levels for plotting
    out["sp500_open"]        = df["Open"].squeeze()
    out["sp500_high"]        = df["High"].squeeze()
    out["sp500_low"]         = df["Low"].squeeze()
    out["sp500_volume"]      = df["Volume"].squeeze()

    return out


def build_vix_features(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich the raw VIX DataFrame."""
    out = pd.DataFrame(index=df.index)
    vix = df["Close"].squeeze()

    out["vix_close"]         = vix
    out["vix_change_1d"]     = vix.diff(1)
    out["vix_pct_change_1d"] = vix.pct_change(1)
    out["vix_ma5"]           = vix.rolling(5).mean()
    out["vix_ma21"]          = vix.rolling(21).mean()

    # VIX term structure proxy: level relative to its own 90-day MA
    out["vix_vs_ma90"]       = vix - vix.rolling(90).mean()

    # VIX regime flag (standard thresholds)
    out["vix_regime"] = pd.cut(
        vix,
        bins=[0, 15, 20, 30, np.inf],
        labels=["low", "moderate", "elevated", "crisis"],
    )

    return out


def main():
    # ── Download raw data ──────────────────────────────────────────────────
    sp500_raw = download(SP500_TICKER, START_DATE, END_DATE)
    vix_raw   = download(VIX_TICKER,   START_DATE, END_DATE)

    # ── Build feature frames ───────────────────────────────────────────────
    sp500 = build_sp500_features(sp500_raw)
    vix   = build_vix_features(vix_raw)

    # ── Save individual files ──────────────────────────────────────────────
    sp500.to_csv(f"{DATA_DIR}/sp500_daily.csv")
    vix.to_csv(f"{DATA_DIR}/vix_daily.csv")
    print(f"[save] {DATA_DIR}/sp500_daily.csv  ({len(sp500)} rows)")
    print(f"[save] {DATA_DIR}/vix_daily.csv    ({len(vix)} rows)")

    # ── Quick sanity check ─────────────────────────────────────────────────
    print("\n── S&P 500 sample ──────────────────────────────────────────────")
    print(sp500[["sp500_close", "sp500_return_1d",
                  "sp500_rv21", "sp500_fwd_1d"]].tail(5).to_string())

    print("\n── VIX sample ──────────────────────────────────────────────────")
    print(vix[["vix_close", "vix_change_1d",
                "vix_ma21", "vix_regime"]].tail(5).to_string())


if __name__ == "__main__":
    main()

[yfinance] Downloading ^GSPC … 501 rows
[yfinance] Downloading ^VIX … 501 rows
[save] C:/Python/CSUREMM/data/sp500_daily.csv  (501 rows)
[save] C:/Python/CSUREMM/data/vix_daily.csv    (501 rows)

── S&P 500 sample ──────────────────────────────────────────────
            sp500_close  sp500_return_1d  sp500_rv21  sp500_fwd_1d
date                                                              
2025-12-23  6909.790039         0.004550    0.104448      0.003221
2025-12-24  6932.049805         0.003221    0.092975     -0.000304
2025-12-26  6929.939941        -0.000304    0.089109     -0.003492
2025-12-29  6905.740234        -0.003492    0.087929     -0.001376
2025-12-30  6896.240234        -0.001376    0.086478           NaN

── VIX sample ──────────────────────────────────────────────────
            vix_close  vix_change_1d   vix_ma21 vix_regime
date                                                      
2025-12-23      14.00      -0.080000  16.387143        low
2025-12-24      13.47      

In [18]:
"""
04_build_panel.py
=================
Merge Kalshi, Google Trends, VIX, and S&P 500 into a single balanced
daily panel aligned on trading days.

Design decisions
----------------
Calendar vs trading days
  We use TRADING DAYS as the spine.  Google Trends and Kalshi may have
  weekend observations; S&P 500 and VIX do not.  We keep the trading-day
  spine and forward-fill Google Trends / Kalshi values over weekends so
  the panel is balanced (no NaN rows from market-closure gaps).

Normalisation
  All input series are already in their natural scales at this stage.
  Standardised (z-score) versions are added with suffix `_z` for the
  regression analyses in later scripts.

Output columns (full list at bottom of file)
  date, [kalshi cols], [trends cols], [sp500 cols], [vix cols],
  plus derived composite columns used in EDA.
"""

import os
import numpy as np
import pandas as pd

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)


def load(path: str, date_col: str = "date") -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df.set_index(date_col)
    df.index = pd.to_datetime(df.index).normalize()
    return df


def trading_day_spine(start: str, end: str) -> pd.DatetimeIndex:
    """Return US business days between start and end (approximate)."""
    return pd.bdate_range(start=start, end=end, freq="B")


def zscore(s: pd.Series) -> pd.Series:
    """Return z-score, ignoring NaN."""
    return (s - s.mean()) / (s.std() + 1e-9)


def main():
    # ── Load component files ───────────────────────────────────────────────
    kalshi = load(f"{DATA_DIR}/kalshi_daily.csv")
    trends = load(f"{DATA_DIR}/google_trends_daily.csv")
    sp500  = load(f"{DATA_DIR}/sp500_daily.csv")
    vix    = load(f"{DATA_DIR}/vix_daily.csv")

    # ── Trading-day spine ──────────────────────────────────────────────────
    start = max(kalshi.index.min(), trends.index.min(),
                sp500.index.min(),  vix.index.min()).strftime("%Y-%m-%d")
    end   = min(kalshi.index.max(), trends.index.max(),
                sp500.index.max(),  vix.index.max()).strftime("%Y-%m-%d")

    spine = pd.DataFrame(index=trading_day_spine(start, end))
    spine.index.name = "date"
    print(f"[panel] Trading-day spine: {start} → {end}  ({len(spine)} days)")

    # ── Join sources ───────────────────────────────────────────────────────
    # Google Trends and Kalshi: forward-fill over weekends/holidays
    panel = (
        spine
        .join(kalshi, how="left")
        .join(trends, how="left", rsuffix="_trends")
        .join(sp500,  how="left")
        .join(vix,    how="left", rsuffix="_vix")
    )

    # Forward-fill non-market sources over non-trading days (they've already
    # been aligned to the spine, but gaps may exist for holidays)
    trend_cols  = [c for c in panel.columns if c in trends.columns]
    kalshi_cols = [c for c in panel.columns if c in kalshi.columns]
    panel[trend_cols  + kalshi_cols] = (
        panel[trend_cols + kalshi_cols].ffill(limit=3)
    )

    # ── Derived features ───────────────────────────────────────────────────
    # Google Trends composite (simple mean of normalised indices)
    trend_signal_cols = [c for c in panel.columns
                         if c in ["federal_reserve", "interest_rate",
                                  "stock_market_crash"]]
    if trend_signal_cols:
        panel["trends_composite"] = panel[trend_signal_cols].mean(axis=1)

    # Kalshi signal: use nearest-meeting probability as the primary signal;
    # fall back to mean probability if nearest is unavailable
    if "kalshi_prob_nearest" in panel.columns:
        panel["kalshi_signal"] = panel["kalshi_prob_nearest"].fillna(
            panel.get("kalshi_prob_mean", np.nan)
        )
    elif "kalshi_prob_mean" in panel.columns:
        panel["kalshi_signal"] = panel["kalshi_prob_mean"]

    # ── Z-score standardisation ────────────────────────────────────────────
    cols_to_z = (
        trend_signal_cols
        + ["trends_composite", "kalshi_signal",
           "kalshi_volume_total", "vix_close",
           "sp500_return_1d", "sp500_rv21"]
    )
    for col in cols_to_z:
        if col in panel.columns:
            panel[f"{col}_z"] = zscore(panel[col])

    # ── Lag features (used in regression) ─────────────────────────────────
    for col in ["trends_composite", "kalshi_signal", "vix_close"]:
        if col in panel.columns:
            panel[f"{col}_lag1"] = panel[col].shift(1)
            panel[f"{col}_lag5"] = panel[col].shift(5)

    # ── Drop rows where ALL market data is missing (true non-trading days) ─
    key_market_cols = ["sp500_close", "vix_close"]
    existing_keys   = [c for c in key_market_cols if c in panel.columns]
    if existing_keys:
        panel = panel.dropna(subset=existing_keys, how="all")

    # ── Save ───────────────────────────────────────────────────────────────
    out_path = f"{DATA_DIR}/panel_daily.csv"
    panel.to_csv(out_path)
    print(f"[save] {out_path}  shape={panel.shape}")

    # ── Coverage report ────────────────────────────────────────────────────
    print("\n── Data Coverage ────────────────────────────────────────────────")
    coverage = (panel.notna().sum() / len(panel) * 100).rename("pct_non_null")
    print(coverage.to_string())

    print("\n── Panel Head ───────────────────────────────────────────────────")
    display_cols = [c for c in [
        "sp500_close", "sp500_return_1d", "vix_close",
        "kalshi_signal", "trends_composite"
    ] if c in panel.columns]
    print(panel[display_cols].head(10).to_string())


if __name__ == "__main__":
    main()

[panel] Trading-day spine: 2024-01-02 → 2025-12-11  (508 days)
[save] data/panel_daily.csv  shape=(489, 47)

── Data Coverage ────────────────────────────────────────────────
kalshi_prob_nearest      100.000000
kalshi_prob_mean         100.000000
kalshi_prob_weighted     100.000000
kalshi_volume_total      100.000000
kalshi_oi_total          100.000000
kalshi_spread_mean       100.000000
n_markets_active         100.000000
nearest_ticker           100.000000
federal_reserve          100.000000
interest_rate            100.000000
stock_market_crash       100.000000
sp500_close              100.000000
sp500_return_1d           99.795501
sp500_return_5d           98.977505
sp500_return_21d          95.705521
sp500_fwd_1d             100.000000
sp500_fwd_5d             100.000000
sp500_rv5                 98.977505
sp500_rv21                95.705521
sp500_open               100.000000
sp500_high               100.000000
sp500_low                100.000000
sp500_volume             100.0000

In [19]:
"""
05_eda.py
=========
Exploratory data analysis for the Kalshi × Google Trends × S&P 500 panel.

Sections
--------
1.  Descriptive statistics
2.  Time-series plots of all signals
3.  Correlation matrix (contemporaneous)
4.  Cross-correlation / lagged correlations (Trends → Kalshi, Trends → SP500)
5.  Rolling correlations to detect regime changes
6.  Distribution diagnostics (histograms + QQ)
7.  Granger causality pre-test (stationarity check via ADF)

All charts saved to outputs/eda/
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"

warnings.filterwarnings("ignore")
os.makedirs(f"{OUTPUTS_DIR}/eda", exist_ok=True)

STYLE = "seaborn-v0_8-whitegrid"
plt.rcParams.update({
    "figure.dpi":   150,
    "font.size":    9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})


def load_panel() -> pd.DataFrame:
    df = pd.read_csv(f"{DATA_DIR}/panel_daily.csv", parse_dates=["date"])
    df = df.set_index("date").sort_index()
    return df


# ── 1. Descriptive statistics ──────────────────────────────────────────────

def section_descriptive(df: pd.DataFrame):
    key_cols = [
        "sp500_close", "sp500_return_1d", "vix_close",
        "kalshi_signal", "kalshi_volume_total",
        "trends_composite",
        "federal_reserve", "interest_rate", "stock_market_crash",
    ]
    cols = [c for c in key_cols if c in df.columns]
    desc = df[cols].describe(percentiles=[.05, .25, .5, .75, .95]).T
    desc["skew"]     = df[cols].skew()
    desc["kurtosis"] = df[cols].kurtosis()

    print("\n══ 1. Descriptive Statistics ═════════════════════════════════")
    print(desc.round(4).to_string())
    desc.to_csv(f"{OUTPUTS_DIR}/eda/descriptive_stats.csv")


# ── 2. Time-series overview plot ──────────────────────────────────────────

def section_timeseries(df: pd.DataFrame):
    with plt.style.context(STYLE):
        fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

        def plot_if(ax, col, label, color, ylabel=""):
            if col in df.columns:
                ax.plot(df.index, df[col], color=color, lw=0.9, label=label)
                ax.set_ylabel(ylabel or label, fontsize=8)
                ax.legend(loc="upper left", fontsize=7)

        plot_if(axes[0], "sp500_close",        "S&P 500",          "#1565c0", "Price")
        plot_if(axes[1], "vix_close",           "VIX",              "#b71c1c", "Index")

        # Kalshi probability
        if "kalshi_signal" in df.columns:
            axes[2].plot(df.index, df["kalshi_signal"],
                         color="#00695c", lw=0.9, label="Kalshi Fed-cut P(YES)")
            axes[2].axhline(0.5, ls="--", lw=0.6, color="grey")
            axes[2].set_ylabel("Probability", fontsize=8)
            axes[2].legend(loc="upper left", fontsize=7)

        # Google Trends
        trend_cols = [c for c in ["federal_reserve", "interest_rate",
                                   "stock_market_crash"] if c in df.columns]
        colors = ["#f57f17", "#e65100", "#4a148c"]
        for col, color in zip(trend_cols, colors):
            axes[3].plot(df.index, df[col], lw=0.8, label=col, color=color)
        axes[3].set_ylabel("Trends Index", fontsize=8)
        axes[3].legend(loc="upper left", fontsize=7)

        plot_if(axes[4], "sp500_return_1d", "S&P 500 Daily Return",
                "#37474f", "Return")
        axes[4].axhline(0, ls="-", lw=0.5, color="black")

        for ax in axes:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

        fig.suptitle("All Signals – Daily Panel Overview", fontsize=11,
                      fontweight="bold", y=1.01)
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/eda/timeseries_overview.png",
                    bbox_inches="tight")
        plt.close(fig)
        print("[EDA] timeseries_overview.png saved")


# ── 3. Correlation matrix ──────────────────────────────────────────────────

def section_correlation(df: pd.DataFrame):
    key_cols = [
        "federal_reserve", "interest_rate", "stock_market_crash",
        "trends_composite", "kalshi_signal", "kalshi_volume_total",
        "vix_close", "sp500_return_1d", "sp500_fwd_1d",
    ]
    cols = [c for c in key_cols if c in df.columns]
    corr = df[cols].corr(method="spearman")  # Spearman for non-normal series

    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(10, 8))
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
        sns.heatmap(
            corr, mask=mask, annot=True, fmt=".2f", linewidths=0.4,
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax, annot_kws={"size": 7},
        )
        ax.set_title("Spearman Correlation Matrix", fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/eda/correlation_matrix.png", bbox_inches="tight")
        plt.close(fig)
        print("[EDA] correlation_matrix.png saved")

    corr.to_csv(f"{OUTPUTS_DIR}/eda/correlation_matrix.csv")


# ── 4. Cross-correlations (Trends/Kalshi → SP500) ─────────────────────────

def section_crosscorr(df: pd.DataFrame):
    """
    Compute cross-correlations at lags 0–10 between:
      a) trends_composite → sp500_fwd_1d
      b) kalshi_signal    → sp500_fwd_1d
      c) trends_composite → kalshi_signal  (information demand → bets)
    """
    pairs = [
        ("trends_composite", "sp500_fwd_1d",  "Trends→SP500(fwd)"),
        ("kalshi_signal",    "sp500_fwd_1d",  "Kalshi→SP500(fwd)"),
        ("trends_composite", "kalshi_signal",  "Trends→Kalshi"),
    ]

    with plt.style.context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        for ax, (x_col, y_col, title) in zip(axes, pairs):
            if x_col not in df.columns or y_col not in df.columns:
                ax.set_visible(False)
                continue

            x = df[x_col].dropna()
            y = df[y_col].dropna()
            idx = x.index.intersection(y.index)
            x, y = x.loc[idx], y.loc[idx]

            lags  = range(-5, 11)   # negative = y leads x
            corrs = [x.corr(y.shift(-lag)) for lag in lags]

            ax.bar(list(lags), corrs, color=["#c62828" if c < 0 else "#1565c0"
                                              for c in corrs], alpha=0.75)
            ax.axhline(0, color="black", lw=0.7)
            ax.axvline(0, color="grey", lw=0.5, ls="--")

            # 95% confidence band (≈ 1.96/√n)
            n    = len(idx)
            band = 1.96 / np.sqrt(n)
            ax.axhline( band, color="red", ls="--", lw=0.7, label="95% CI")
            ax.axhline(-band, color="red", ls="--", lw=0.7)

            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Lag (days, positive = x leads y)")
            ax.set_ylabel("Correlation")
            ax.legend(fontsize=7)

        fig.suptitle("Cross-Correlations at Lags -5 to +10", fontsize=10,
                      fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/eda/cross_correlations.png",
                    bbox_inches="tight")
        plt.close(fig)
        print("[EDA] cross_correlations.png saved")


# ── 5. Rolling correlations ────────────────────────────────────────────────

def section_rolling_corr(df: pd.DataFrame, window: int = 63):
    """21-trading-day (≈ 63-day ≈ 3-month) rolling Pearson correlations."""
    pairs = [
        ("trends_composite", "vix_close",       "Trends vs VIX"),
        ("trends_composite", "sp500_return_1d",  "Trends vs SP500 return"),
        ("kalshi_signal",    "vix_close",        "Kalshi vs VIX"),
        ("kalshi_signal",    "sp500_return_1d",  "Kalshi vs SP500 return"),
    ]

    with plt.style.context(STYLE):
        fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
        axes = axes.flatten()

        for ax, (c1, c2, title) in zip(axes, pairs):
            if c1 not in df.columns or c2 not in df.columns:
                ax.set_visible(False)
                continue
            roll = df[c1].rolling(window).corr(df[c2])
            ax.plot(df.index, roll, lw=0.9, color="#0d47a1")
            ax.axhline(0, color="black", lw=0.5)
            ax.fill_between(df.index, roll, 0,
                             where=(roll >= 0), alpha=0.2, color="#1565c0")
            ax.fill_between(df.index, roll, 0,
                             where=(roll < 0), alpha=0.2, color="#b71c1c")
            ax.set_title(f"{window}-day Rolling Corr: {title}", fontsize=9)
            ax.set_ylabel("Pearson r")
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

        fig.suptitle(f"Rolling {window}-day Correlations", fontsize=10,
                      fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/eda/rolling_correlations.png",
                    bbox_inches="tight")
        plt.close(fig)
        print("[EDA] rolling_correlations.png saved")


# ── 6. Distribution diagnostics ───────────────────────────────────────────

def section_distributions(df: pd.DataFrame):
    cols = [c for c in ["sp500_return_1d", "vix_close",
                         "kalshi_signal", "trends_composite"]
            if c in df.columns]

    with plt.style.context(STYLE):
        fig, axes = plt.subplots(2, len(cols), figsize=(4 * len(cols), 7))

        for i, col in enumerate(cols):
            s = df[col].dropna()

            # Histogram + KDE
            axes[0, i].hist(s, bins=50, density=True, alpha=0.6, color="#1565c0")
            s.plot.kde(ax=axes[0, i], color="#b71c1c", lw=1.2)
            axes[0, i].set_title(col, fontsize=8)

            # QQ plot
            stats.probplot(s, dist="norm", plot=axes[1, i])
            axes[1, i].set_title(f"QQ: {col}", fontsize=8)

        fig.suptitle("Distribution Diagnostics", fontsize=10, fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/eda/distributions.png", bbox_inches="tight")
        plt.close(fig)
        print("[EDA] distributions.png saved")


# ── 7. Stationarity (ADF) and Granger pre-test ────────────────────────────

def section_stationarity(df: pd.DataFrame):
    cols = [c for c in ["federal_reserve", "interest_rate",
                         "stock_market_crash", "trends_composite",
                         "kalshi_signal", "vix_close", "sp500_return_1d"]
            if c in df.columns]

    results = []
    for col in cols:
        s = df[col].dropna()
        adf_stat, p_val, lags, *_ = adfuller(s, autolag="AIC")
        results.append({
            "series":    col,
            "adf_stat":  round(adf_stat, 4),
            "p_value":   round(p_val,    4),
            "lags":      lags,
            "stationary": "✓" if p_val < 0.05 else "✗  (needs differencing)",
        })

    res_df = pd.DataFrame(results)
    print("\n══ 7. ADF Stationarity Tests ═════════════════════════════════")
    print(res_df.to_string(index=False))
    res_df.to_csv(f"{OUTPUTS_DIR}/eda/adf_tests.csv", index=False)


def section_granger(df: pd.DataFrame, max_lag: int = 5):
    """Quick bivariate Granger causality: trends_composite → sp500_return_1d."""
    pairs = [
        ("trends_composite", "sp500_return_1d"),
        ("kalshi_signal",    "sp500_return_1d"),
        ("trends_composite", "kalshi_signal"),
    ]
    print("\n══ Granger Causality Tests (max_lag={}) ══════════════════════".format(max_lag))

    for x_col, y_col in pairs:
        if x_col not in df.columns or y_col not in df.columns:
            continue
        data = df[[y_col, x_col]].dropna()
        if len(data) < max_lag + 10:
            print(f"  {x_col}→{y_col}: insufficient data")
            continue
        try:
            res = grangercausalitytests(data, maxlag=max_lag, verbose=False)
            # Report minimum p-value across lags (ssr_ftest)
            min_p = min(
                res[lag][0]["ssr_ftest"][1]
                for lag in range(1, max_lag + 1)
            )
            verdict = "✓ significant" if min_p < 0.05 else "✗ not significant"
            print(f"  {x_col} → {y_col}: min p={min_p:.4f}  {verdict}")
        except Exception as e:
            print(f"  {x_col}→{y_col}: error – {e}")


# ── Main ───────────────────────────────────────────────────────────────────

def main():
    df = load_panel()
    print(f"[EDA] Panel loaded: {df.shape[0]} rows × {df.shape[1]} cols")

    section_descriptive(df)
    section_timeseries(df)
    section_correlation(df)
    section_crosscorr(df)
    section_rolling_corr(df)
    section_distributions(df)
    section_stationarity(df)
    section_granger(df)

    print(f"\n[EDA] All outputs in {OUTPUTS_DIR}/eda/")


if __name__ == "__main__":
    main()

Matplotlib is building the font cache; this may take a moment.


[EDA] Panel loaded: 489 rows × 47 cols

══ 1. Descriptive Statistics ═════════════════════════════════
                     count        mean         std        min         5%        25%         50%         75%          95%          max     skew  kurtosis
sp500_close          489.0   5793.5767    556.3880  4688.6802  4955.9819  5346.9902   5780.0498   6115.0698    6748.3440    6901.0000   0.1763   -0.8030
sp500_return_1d      488.0      0.0008      0.0101    -0.0597    -0.0157    -0.0032      0.0011      0.0057       0.0136       0.0952   0.7138   18.7478
vix_close            489.0     17.3346      4.7942    11.8600    12.6000    14.4700     16.3500     18.7100      24.8580      52.3300   2.8564   13.0164
kalshi_signal        489.0      0.2936      0.4051     0.0100     0.0100     0.0100      0.0200      0.8100       0.9800       0.9900   0.9142   -1.0565
kalshi_volume_total  489.0  38606.8834  55564.5408     1.0000   318.6000  6496.0000  17326.0000  45588.0000  146545.0000  441121.000

In [23]:
"""
06_trends_kalshi_relationship.py
=================================
Test the core hypothesis:

    H1: Google Trends (information *demand*) Granger-causes Kalshi prices
        (information *aggregation*), with lags of 1–5 days.

    H2: Trends and Kalshi together explain more S&P 500 return variation
        than either alone.

    H3: The information-demand → information-aggregation channel is
        stronger during high-volatility (VIX > 20) regimes.

Methods
-------
  A. OLS / WLS regression with Newey-West SEs (heteroskedasticity + autocorr)
  B. VAR(p) model for Trends ↔ Kalshi ↔ SP500 returns
  C. Regime-split analysis (low / high VIX)
  D. Scatter plots + partial regression plots

Dependencies: pip install statsmodels scipy pandas matplotlib seaborn
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.stats.sandwich_covariance import cov_hac
import statsmodels.api as sm

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"

warnings.filterwarnings("ignore")
os.makedirs(f"{OUTPUTS_DIR}/analysis", exist_ok=True)

STYLE = "seaborn-v0_8-whitegrid"


def load_panel() -> pd.DataFrame:
    df = pd.read_csv(f"{DATA_DIR}/panel_daily.csv",
                     parse_dates=["date"], index_col="date")
    return df.sort_index()


def maybe_difference(s: pd.Series, name: str) -> pd.Series:
    """First-difference if ADF rejects stationarity at 10% level."""
    _, p, *_ = adfuller(s.dropna(), autolag="AIC")
    if p > 0.10:
        print(f"  [diff] {name} is non-stationary (p={p:.3f}); first-differencing.")
        return s.diff().dropna()
    return s


# ── A. OLS with Newey-West SEs ─────────────────────────────────────────────

def ols_newey_west(df: pd.DataFrame):
    """
    Regress kalshi_signal on lagged trends (1d and 5d lags) + controls.
    Robust SEs via Newey-West (HAC) to handle autocorrelation.
    """
    cols_needed = ["kalshi_signal", "trends_composite",
                   "vix_close", "sp500_return_1d"]
    data = df[[c for c in cols_needed if c in df.columns]].dropna().copy()

    data["trends_lag1"] = data["trends_composite"].shift(1)
    data["trends_lag3"] = data["trends_composite"].shift(3)
    data["trends_lag5"] = data["trends_composite"].shift(5)
    data["vix_lag1"]    = data["vix_close"].shift(1)
    data["sp500_lag1"]  = data["sp500_return_1d"].shift(1)
    data = data.dropna()

    y = data["kalshi_signal"]
    X = add_constant(data[["trends_lag1", "trends_lag3", "trends_lag5",
                             "vix_lag1", "sp500_lag1"]])

    model  = OLS(y, X).fit()
    # Newey-West (HAC) covariance with 5-lag bandwidth
    hac_se = cov_hac(model, nlags=5)
    model_hac = model.get_robustcov_results(cov_type="HAC",
                                             use_t=True, maxlags=5)

    print("\n══ A. OLS: Kalshi ~ Trends_Lagged + Controls ════════════════")
    print(model_hac.summary())
    with open(f"{OUTPUTS_DIR}/analysis/ols_kalshi_on_trends.txt", "w") as f:
        f.write(str(model_hac.summary()))

    return model_hac, data


# ── B. VAR model ───────────────────────────────────────────────────────────

def var_model(df: pd.DataFrame):
    """
    Fit a VAR(p) on [trends_composite, kalshi_signal, sp500_return_1d].
    Select lag order via AIC.  Extract Granger causality tests and
    impulse response functions (IRFs).
    """
    var_cols = [c for c in ["trends_composite", "kalshi_signal",
                              "sp500_return_1d"] if c in df.columns]
    data = df[var_cols].dropna().copy()

    # Make stationary
    stationary = pd.DataFrame(index=data.index)
    for col in var_cols:
        stationary[col] = maybe_difference(data[col], col)
    stationary = stationary.dropna()

    print("\n══ B. VAR Model ═════════════════════════════════════════════")
    model  = VAR(stationary)
    # Lag selection: check up to 10 lags
    lag_results = model.select_order(maxlags=10)
    best_lag    = max(lag_results.selected_orders["aic"], 1)         # at least 1
    print(f"  AIC-selected lag order: {best_lag}")

    res = model.fit(best_lag)
    print(res.summary())

    # ── Granger causality from VAR ─────────────────────────────────────────
    print("\n── VAR Granger Causality ─────────────────────────────────────")
    for caused in var_cols:
        for causing in [c for c in var_cols if c != caused]:
            try:
                gc = res.test_causality(caused, causing, kind="f")
                print(f"  {causing} → {caused}: "
                      f"F={gc.test_statistic:.3f}, p={gc.pvalue:.4f} "
                      f"{'✓' if gc.pvalue < 0.05 else '✗'}")
            except Exception as e:
                print(f"  {causing}→{caused}: {e}")

    # ── Impulse Response Functions ─────────────────────────────────────────
    irf = res.irf(10)
    with plt.style.context(STYLE):
        fig = irf.plot(orth=True, figsize=(12, 8))
        fig.suptitle("Orthogonalised IRF – VAR(p)", fontsize=10, fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/analysis/var_irf.png", bbox_inches="tight")
        plt.close(fig)
        print(f"[save] var_irf.png")

    return res, stationary


# ── C. Regime-split analysis ───────────────────────────────────────────────

def regime_analysis(df: pd.DataFrame, ols_data: pd.DataFrame):
    """
    Repeat the OLS regression separately for:
       Low VIX  (<= 20)
       High VIX (>  20)
    to test whether the Trends→Kalshi channel is stronger during stress.
    """
    print("\n══ C. Regime-Split Regressions (VIX < 20 vs ≥ 20) ═════════")
    regimes = {
        "low_vix":  ols_data[ols_data["vix_close"] <= 20],
        "high_vix": ols_data[ols_data["vix_close"] >  20],
    }

    summary_rows = []
    for regime, sub in regimes.items():
        if len(sub) < 20:
            print(f"  {regime}: insufficient data ({len(sub)} rows)")
            continue
        y = sub["kalshi_signal"]
        X = add_constant(sub[["trends_lag1", "trends_lag3", "trends_lag5",
                                "vix_lag1",   "sp500_lag1"]])
        m = OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        print(f"\n  [{regime}]  n={len(sub)}  R²={m.rsquared:.4f}")
        for name, coef, pval in zip(m.params.index, m.params, m.pvalues):
            sig = "**" if pval < 0.05 else ("*" if pval < 0.10 else "")
            print(f"    {name:<20} {coef:+.4f}  p={pval:.3f} {sig}")
            summary_rows.append({"regime": regime, "var": name,
                                   "coef": coef, "pval": pval})

    pd.DataFrame(summary_rows).to_csv(
        f"{OUTPUTS_DIR}/analysis/regime_regression.csv", index=False
    )


# ── D. Scatter / partial regression plots ─────────────────────────────────

def scatter_plots(df: pd.DataFrame):
    pairs = [
        ("trends_composite", "kalshi_signal",    "Trends→Kalshi"),
        ("kalshi_signal",    "sp500_fwd_1d",     "Kalshi→SP500(fwd)"),
        ("trends_composite", "sp500_fwd_1d",     "Trends→SP500(fwd)"),
    ]

    with plt.style.context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        for ax, (xc, yc, title) in zip(axes, pairs):
            if xc not in df.columns or yc not in df.columns:
                ax.set_visible(False)
                continue
            data = df[[xc, yc]].dropna()
            ax.scatter(data[xc], data[yc], alpha=0.3, s=8, color="#1565c0")

            # OLS trend line
            x_vals = np.linspace(data[xc].min(), data[xc].max(), 200)
            coef   = np.polyfit(data[xc], data[yc], 1)
            ax.plot(x_vals, np.polyval(coef, x_vals),
                    color="#b71c1c", lw=1.5, label=f"slope={coef[0]:.3f}")

            r, p = pd.Series(data[xc]).corr(pd.Series(data[yc])), \
                   stats_corr(data[xc], data[yc])
            ax.set_title(f"{title}\nr={r:.3f}  p={p:.3f}", fontsize=9)
            ax.set_xlabel(xc, fontsize=8)
            ax.set_ylabel(yc, fontsize=8)
            ax.legend(fontsize=7)

        fig.suptitle("Pairwise Scatter Plots", fontsize=10, fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/analysis/scatter_plots.png", bbox_inches="tight")
        plt.close(fig)
        print("[save] scatter_plots.png")


def stats_corr(x, y):
    from scipy.stats import pearsonr
    _, p = pearsonr(x, y)
    return p


# ── Main ───────────────────────────────────────────────────────────────────

def main():
    df = load_panel()
    print(f"[analysis] Panel: {df.shape}")

    model_hac, ols_data = ols_newey_west(df)
    var_res, stat_data  = var_model(df)
    regime_analysis(df, ols_data)
    scatter_plots(df)

    print(f"\n[analysis] All outputs in {OUTPUTS_DIR}/analysis/")


if __name__ == "__main__":
    main()

[analysis] Panel: (489, 47)

══ A. OLS: Kalshi ~ Trends_Lagged + Controls ════════════════
                            OLS Regression Results                            
Dep. Variable:          kalshi_signal   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.053
Method:                 Least Squares   F-statistic:                     4.966
Date:                Sat, 13 Jun 2026   Prob (F-statistic):           0.000192
Time:                        22:31:00   Log-Likelihood:                -228.42
No. Observations:                 483   AIC:                             468.8
Df Residuals:                     477   BIC:                             493.9
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------

In [26]:
"""
07_fear_greed_index.py
======================
Construct a "Fear & Greed" index from Google Trends and Kalshi data.

Index design philosophy
------------------------
Both Google Trends and Kalshi carry complementary information:

  • Google Trends  → DEMAND for information about macro risks.
    High "stock market crash" / "interest rate" searches = more FEAR.
    (Positive values → Fear; the series is a raw anxiety signal.)

  • Kalshi (KXFED) → AGGREGATED probability of a Fed rate cut.
    High P(cut) → market expects accommodation → GREED/complacency.
    Low P(cut)  → tightening risk looms → FEAR.

The composite index is constructed in two complementary ways:

  Method 1 – PCA-weighted composite (data-driven)
    Run PCA on [trends_fear, kalshi_cut_prob_inverted, vix_normalised].
    Use PC1 loadings as weights. Sign-normalise so higher = more Fear.

  Method 2 – Equal-weight composite (interpretable)
    Normalize each sub-index to 0–100:
      • trends_fear_score   : 0.50 × (federal_reserve_z
                                       + interest_rate_z
                                       + stock_market_crash_z)
      • kalshi_fear_score   : 0.30 × (1 - kalshi_prob_cut_normed)
                              because HIGH cut probability → less fear
      • vix_fear_score      : 0.20 × vix_normed
    Sum and re-scale to 0–100.
    0–25   = Extreme Greed
    25–45  = Greed
    45–55  = Neutral
    55–75  = Fear
    75–100 = Extreme Fear

Both methods saved; the PCA index is the recommended research index.

Output
------
  data/fear_greed_index.csv  – daily Fear & Greed values + regime labels
  outputs/fear_greed/        – all charts
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"

warnings.filterwarnings("ignore")
os.makedirs(f"{OUTPUTS_DIR}/fear_greed", exist_ok=True)

STYLE = "seaborn-v0_8-whitegrid"
# Colour map: green (greed) → yellow (neutral) → red (fear)
FG_CMAP = LinearSegmentedColormap.from_list(
    "fear_greed",
    ["#1b5e20", "#a5d6a7", "#fffde7", "#ef9a9a", "#b71c1c"]
)


def load_panel() -> pd.DataFrame:
    df = pd.read_csv(f"{DATA_DIR}/panel_daily.csv",
                     parse_dates=["date"], index_col="date")
    return df.sort_index()


def minmax(s: pd.Series) -> pd.Series:
    """Clip extremes at 2% / 98% quantile then scale to 0–100."""
    lo, hi = s.quantile(0.02), s.quantile(0.98)
    s = s.clip(lo, hi)
    rng = hi - lo
    if rng < 1e-9:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - lo) / rng


def label_regime(score: float) -> str:
    if score < 25:   return "Extreme Greed"
    if score < 45:   return "Greed"
    if score < 55:   return "Neutral"
    if score < 75:   return "Fear"
    return "Extreme Fear"


# ── Sub-index construction ─────────────────────────────────────────────────

def build_sub_indices(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    # 1. Trends Fear Score:
    #    Average of the three search terms (already 0-100 normalised).
    #    "stock market crash" weighted 2× because it is the most explicit fear signal.
    trend_cols = [c for c in ["federal_reserve", "interest_rate",
                               "stock_market_crash"] if c in df.columns]
    if trend_cols:
        weights    = {"federal_reserve": 1.0, "interest_rate": 1.0,
                      "stock_market_crash": 2.0}
        wt_sum     = sum(weights.get(c, 1.0) for c in trend_cols)
        out["trends_fear_raw"] = sum(
            df[c] * weights.get(c, 1.0) for c in trend_cols
        ) / wt_sum
        out["trends_fear"] = minmax(out["trends_fear_raw"])
    else:
        out["trends_fear"] = np.nan

    # 2. Kalshi Fear Score:
    #    INVERT the cut-probability: high P(cut) = accommodation = less fear.
    if "kalshi_signal" in df.columns:
        # kalshi_signal is 0–1 probability; scale to 0–100 then invert
        ks = minmax(df["kalshi_signal"])
        out["kalshi_fear"] = 100 - ks   # invert: high cut prob → low fear
    else:
        out["kalshi_fear"] = np.nan

    # 3. VIX Fear Score
    if "vix_close" in df.columns:
        out["vix_fear"] = minmax(df["vix_close"])
    else:
        out["vix_fear"] = np.nan

    return out


# ── Method 1: Equal-weight composite ──────────────────────────────────────

def equal_weight_index(sub: pd.DataFrame) -> pd.Series:
    weights = {"trends_fear": 0.50, "kalshi_fear": 0.30, "vix_fear": 0.20}
    avail   = {k: v for k, v in weights.items() if k in sub.columns}
    total_w = sum(avail.values())

    fg = sum(sub[k] * v for k, v in avail.items()) / total_w
    fg.name = "fg_equal_weight"
    return fg


# ── Method 2: PCA-weighted composite ──────────────────────────────────────

def pca_index(sub: pd.DataFrame) -> pd.Series:
    cols  = [c for c in ["trends_fear", "kalshi_fear", "vix_fear"]
             if c in sub.columns and sub[c].notna().sum() > 10]
    data  = sub[cols].dropna()

    scaler = StandardScaler()
    scaled = scaler.fit_transform(data)

    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(scaled).flatten()

    print(f"\n[PCA] Explained variance (PC1): {pca.explained_variance_ratio_[0]:.1%}")
    for col, loading in zip(cols, pca.components_[0]):
        print(f"  {col:<20} loading={loading:+.4f}")

    # Sign convention: ensure VIX has a positive loading (fear = high VIX)
    vix_idx = cols.index("vix_fear") if "vix_fear" in cols else 0
    if pca.components_[0][vix_idx] < 0:
        pc1 = -pc1

    pc1_series = pd.Series(pc1, index=data.index, name="fg_pca_raw")

    # Map onto 0-100
    fg_pca = pd.Series(
        minmax(pc1_series), index=data.index, name="fg_pca"
    )
    return fg_pca


# ── Plotting ───────────────────────────────────────────────────────────────

def plot_index(fg: pd.Series, sp500: pd.Series, vix: pd.Series,
               label: str, filename: str):
    with plt.style.context(STYLE):
        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1.5, 1.5]})

        # Panel 1: Fear & Greed gauge as a filled time series
        ax = axes[0]
        x  = fg.index
        y  = fg.values
        ax.fill_between(x, 50, y, where=(y >= 50), alpha=0.7, color="#c62828",
                         label="Fear zone")
        ax.fill_between(x, 50, y, where=(y < 50), alpha=0.7, color="#1b5e20",
                         label="Greed zone")
        ax.plot(x, y, lw=0.7, color="black")
        for thresh, lbl in [(25, "Ext Greed"), (45, "Greed"),
                             (55, "Fear"),     (75, "Ext Fear")]:
            ax.axhline(thresh, ls="--", lw=0.5, color="grey")
            ax.text(x[-1], thresh + 0.5, lbl, fontsize=6, color="grey",
                    ha="right")
        ax.set_ylim(0, 100)
        ax.set_ylabel("Fear & Greed (0=Greed, 100=Fear)")
        ax.set_title(f"Fear & Greed Index  [{label}]",
                      fontsize=11, fontweight="bold")
        ax.legend(loc="upper left", fontsize=7)

        # Panel 2: S&P 500
        ax2 = axes[1]
        if sp500 is not None and not sp500.empty:
            ax2.plot(sp500.index, sp500.values, lw=0.9, color="#1565c0")
        ax2.set_ylabel("S&P 500")

        # Panel 3: VIX
        ax3 = axes[2]
        if vix is not None and not vix.empty:
            ax3.plot(vix.index, vix.values, lw=0.9, color="#b71c1c")
            ax3.axhline(20, ls="--", lw=0.6, color="grey", label="VIX=20")
            ax3.legend(fontsize=7)
        ax3.set_ylabel("VIX")

        for ax in axes:
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/fear_greed/{filename}", bbox_inches="tight")
        plt.close(fig)
        print(f"[save] {filename}")


def plot_sub_indices(sub: pd.DataFrame):
    """Show how each sub-index contributes."""
    cols = [c for c in ["trends_fear", "kalshi_fear", "vix_fear"]
            if c in sub.columns]
    colors = {"trends_fear": "#f57f17", "kalshi_fear": "#4a148c",
               "vix_fear": "#b71c1c"}

    with plt.style.context(STYLE):
        fig, axes = plt.subplots(len(cols), 1, figsize=(14, 3 * len(cols)),
                                  sharex=True)
        if len(cols) == 1:
            axes = [axes]

        for ax, col in zip(axes, cols):
            ax.fill_between(sub.index, 0, sub[col],
                             alpha=0.6, color=colors.get(col, "steelblue"))
            ax.plot(sub.index, sub[col], lw=0.7,
                    color=colors.get(col, "steelblue"))
            ax.axhline(50, ls="--", lw=0.5, color="grey")
            ax.set_ylabel(col, fontsize=8)
            ax.set_ylim(0, 100)
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

        fig.suptitle("Fear & Greed Sub-Indices (0=Greed, 100=Fear)",
                      fontsize=10, fontweight="bold")
        plt.tight_layout()
        fig.savefig(f"{OUTPUTS_DIR}/fear_greed/sub_indices.png", bbox_inches="tight")
        plt.close(fig)
        print("[save] sub_indices.png")


# ── Main ───────────────────────────────────────────────────────────────────

def main():
    df = load_panel()

    sub  = build_sub_indices(df)
    fg_ew  = equal_weight_index(sub)
    fg_pca = pca_index(sub)

    # Combine into output frame
    out = sub.copy()
    out["fg_equal_weight"] = fg_ew
    if fg_pca is not None:
        out = out.join(fg_pca, how="left")
    else:
        out["fg_pca"] = np.nan

    # Add regime labels (primary: PCA index)
    primary = out["fg_pca"].fillna(out["fg_equal_weight"])
    out["fg_primary"]        = primary
    out["fg_regime"]         = primary.map(label_regime)

    # Attach sp500 and vix for context
    out["sp500_close"]       = df.get("sp500_close")
    out["sp500_return_1d"]   = df.get("sp500_return_1d")
    out["vix_close"]         = df.get("vix_close")

    out.to_csv(f"{DATA_DIR}/fear_greed_index.csv")
    print(f"\n[save] {DATA_DIR}/fear_greed_index.csv  ({len(out)} rows)")

    # Regime distribution
    print("\n── Regime Distribution ─────────────────────────────────────────")
    print(out["fg_regime"].value_counts().to_string())

    # Plots
    sp500 = df["sp500_close"] if "sp500_close" in df.columns else None
    vix   = df["vix_close"]   if "vix_close"   in df.columns else None

    plot_index(fg_ew, sp500, vix,  "Equal-Weight", "fg_equal_weight.png")
    if "fg_pca" in out.columns:
        plot_index(out["fg_pca"].dropna(), sp500, vix,
                   "PCA-Weighted", "fg_pca.png")
    plot_sub_indices(sub)

    print(f"\n[fear_greed] Outputs in {OUTPUTS_DIR}/fear_greed/")


if __name__ == "__main__":
    main()


[PCA] Explained variance (PC1): 47.9%
  trends_fear          loading=+0.6364
  kalshi_fear          loading=+0.4582
  vix_fear             loading=+0.6206

[save] data/fear_greed_index.csv  (489 rows)

── Regime Distribution ─────────────────────────────────────────
fg_regime
Greed            196
Extreme Greed    123
Neutral           68
Fear              61
Extreme Fear      41
[save] fg_equal_weight.png
[save] fg_pca.png
[save] sub_indices.png

[fear_greed] Outputs in outputs/fear_greed/


In [27]:
"""
08_fg_vs_vix_performance.py
============================
Compare the Fear & Greed index against VIX as predictors of S&P 500
returns across three dimensions:

  A. Predictive regressions (R², coefficients, Newey-West SEs)
  B. Directional accuracy (hit rate: does signal correctly call up/down?)
  C. Regime-based trading backtest (Sharpe, max drawdown, CAGR)
  D. Summary comparison chart

Backtest rules
--------------
  FG strategy : long S&P when fg_primary < 55 (Neutral/Greed),
                flat (cash, 0% return) when fg_primary >= 55 (Fear)
  VIX strategy: long S&P when vix_close  < 20,
                flat when vix_close >= 20
  Buy-and-hold: always long S&P

Signal is lagged 1 day (today's close → tomorrow's position) to avoid
look-ahead bias.  No transaction costs modelled (MVP).
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "data"
OUTPUTS_DIR = "outputs"

warnings.filterwarnings("ignore")
os.makedirs(f"{OUTPUTS_DIR}/performance", exist_ok=True)

STYLE = "seaborn-v0_8-whitegrid"


# ── Load ───────────────────────────────────────────────────────────────────

def load() -> pd.DataFrame:
    fg  = pd.read_csv(f"{DATA_DIR}/fear_greed_index.csv",
                      parse_dates=["date"], index_col="date")
    pan = pd.read_csv(f"{DATA_DIR}/panel_daily.csv",
                      parse_dates=["date"], index_col="date")

    df = pan[["sp500_close", "sp500_return_1d",
              "sp500_fwd_1d", "sp500_fwd_5d",
              "vix_close"]].join(
         fg[["fg_primary", "fg_equal_weight", "fg_pca", "fg_regime"]],
         how="inner"
    ).sort_index().dropna(subset=["sp500_return_1d", "vix_close", "fg_primary"])

    print(f"[load] {len(df)} rows after join")
    return df


# ── A. Predictive regressions ──────────────────────────────────────────────

def regression_comparison(df: pd.DataFrame) -> pd.DataFrame:
    """
    Regress sp500_fwd_1d on:
      Model 1 : VIX only
      Model 2 : FG only
      Model 3 : VIX + FG (joint)
    Report R², adj-R², coefs, HAC p-values.
    """
    results = []
    data = df[["sp500_fwd_1d", "vix_close", "fg_primary"]].dropna().copy()

    # Standardise predictors for comparable coefficients
    data["vix_z"] = (data["vix_close"]  - data["vix_close"].mean())  / data["vix_close"].std()
    data["fg_z"]  = (data["fg_primary"] - data["fg_primary"].mean()) / data["fg_primary"].std()

    specs = {
        "VIX only":  ["vix_z"],
        "FG only":   ["fg_z"],
        "VIX + FG":  ["vix_z", "fg_z"],
    }

    print("\n══ A. Predictive Regressions (DV = sp500_fwd_1d) ═══════════════")
    for label, predictors in specs.items():
        X = sm.add_constant(data[predictors])
        y = data["sp500_fwd_1d"]
        m = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

        row = {
            "model":    label,
            "n":        int(m.nobs),
            "R2":       round(m.rsquared,      4),
            "adj_R2":   round(m.rsquared_adj,  4),
            "F_pval":   round(m.f_pvalue,      4),
        }
        for var in predictors:
            row[f"coef_{var}"] = round(m.params[var],  5)
            row[f"pval_{var}"] = round(m.pvalues[var], 4)

        results.append(row)
        print(f"\n  [{label}]  R²={row['R2']}  adj-R²={row['adj_R2']}  "
              f"F-p={row['F_pval']}")
        for var in predictors:
            sig = "**" if row[f"pval_{var}"] < 0.05 else (
                  "*"  if row[f"pval_{var}"] < 0.10 else "")
            print(f"    {var:<10} coef={row[f'coef_{var}']:+.5f}  "
                  f"p={row[f'pval_{var}']:.4f} {sig}")

    res_df = pd.DataFrame(results)
    res_df.to_csv(f"{OUTPUTS_DIR}/performance/regression_comparison.csv",
                  index=False)
    return res_df


# ── B. Directional accuracy ────────────────────────────────────────────────

def directional_accuracy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hit rate: does each signal correctly predict the DIRECTION of next-day return?

    FG signal  : Fear (>=55) → predict DOWN; Greed/Neutral (<55) → predict UP
    VIX signal : Elevated (>=20) → predict DOWN; Low (<20) → predict UP
    """
    data = df[["sp500_fwd_1d", "vix_close", "fg_primary"]].dropna().copy()

    data["actual_up"]   = data["sp500_fwd_1d"] > 0
    data["fg_pred_up"]  = data["fg_primary"]  < 55   # Greed → long
    data["vix_pred_up"] = data["vix_close"]   < 20   # Low VIX → long

    rows = []
    for label, pred_col in [("FG Index", "fg_pred_up"),
                              ("VIX",      "vix_pred_up")]:
        correct   = (data["actual_up"] == data[pred_col]).mean()
        # Subset: only days where signal IS directional (not neutral)
        # For VIX: all days have a direction; for FG same
        # Chi-square test of independence
        ct = pd.crosstab(data[pred_col], data["actual_up"])
        try:
            chi2, p_chi, *_ = stats.chi2_contingency(ct)
        except Exception:
            chi2, p_chi = np.nan, np.nan

        rows.append({
            "signal":        label,
            "hit_rate":      round(correct, 4),
            "n":             len(data),
            "chi2":          round(chi2, 3) if not np.isnan(chi2) else np.nan,
            "p_chi2":        round(p_chi, 4) if not np.isnan(p_chi) else np.nan,
            "significant":   "✓" if (not np.isnan(p_chi) and p_chi < 0.05) else "✗",
        })

    res_df = pd.DataFrame(rows)
    print("\n══ B. Directional Accuracy ══════════════════════════════════════")
    print(res_df.to_string(index=False))
    res_df.to_csv(f"{OUTPUTS_DIR}/performance/directional_accuracy.csv",
                  index=False)
    return res_df


# ── C. Regime-based backtest ───────────────────────────────────────────────

def backtest(df: pd.DataFrame) -> pd.DataFrame:
    """
    Simple daily rebalancing backtest.
    Position is determined by PREVIOUS day's signal (no look-ahead).
    """
    data = df[["sp500_return_1d", "vix_close", "fg_primary"]].dropna().copy()

    # Signals (lagged 1 day)
    data["fg_long"]  = (data["fg_primary"].shift(1)  < 55).astype(float)
    data["vix_long"] = (data["vix_close"].shift(1)   < 20).astype(float)
    data["bah_long"] = 1.0   # buy-and-hold baseline

    data = data.dropna()

    strategies = {
        "Buy & Hold": "bah_long",
        "VIX (<20)":  "vix_long",
        "FG (<55)":   "fg_long",
    }

    equity = pd.DataFrame(index=data.index)
    metrics_rows = []

    for name, pos_col in strategies.items():
        ret = data["sp500_return_1d"] * data[pos_col]
        eq  = (1 + ret).cumprod()
        equity[name] = eq

        total_ret  = eq.iloc[-1] - 1
        n_years    = len(data) / 252
        cagr       = (1 + total_ret) ** (1 / n_years) - 1
        ann_vol    = ret.std() * np.sqrt(252)
        sharpe     = (ret.mean() * 252) / (ann_vol + 1e-9)
        drawdown   = (eq / eq.cummax() - 1)
        max_dd     = drawdown.min()
        days_long  = int(data[pos_col].sum())
        pct_long   = data[pos_col].mean()

        metrics_rows.append({
            "strategy":   name,
            "total_ret":  f"{total_ret:.1%}",
            "CAGR":       f"{cagr:.1%}",
            "ann_vol":    f"{ann_vol:.1%}",
            "Sharpe":     f"{sharpe:.3f}",
            "max_DD":     f"{max_dd:.1%}",
            "days_long":  days_long,
            "pct_long":   f"{pct_long:.1%}",
        })

    metrics_df = pd.DataFrame(metrics_rows)
    print("\n══ C. Backtest Results ══════════════════════════════════════════")
    print(metrics_df.to_string(index=False))
    metrics_df.to_csv(f"{OUTPUTS_DIR}/performance/backtest_metrics.csv",
                      index=False)

    return equity, metrics_df


# ── D. Summary chart ───────────────────────────────────────────────────────

def plot_summary(df: pd.DataFrame, equity: pd.DataFrame,
                 reg_df: pd.DataFrame, dir_df: pd.DataFrame,
                 met_df: pd.DataFrame):

    colors = {
        "Buy & Hold": "#455a64",
        "VIX (<20)":  "#b71c1c",
        "FG (<55)":   "#1565c0",
    }

    with plt.style.context(STYLE):
        fig = plt.figure(figsize=(16, 14))
        gs  = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.35)

        # ── Panel 1: Equity curves ─────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0, :])
        for col in equity.columns:
            ax1.plot(equity.index, equity[col],
                     label=col, color=colors.get(col, "grey"), lw=1.2)
        ax1.set_title("Equity Curves – FG vs VIX Regime Strategy vs Buy & Hold",
                       fontweight="bold")
        ax1.set_ylabel("Portfolio Value (start=1)")
        ax1.legend(fontsize=8)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha="right")

        # ── Panel 2: Drawdowns ────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, :])
        for col in equity.columns:
            dd = (equity[col] / equity[col].cummax() - 1) * 100
            ax2.fill_between(equity.index, dd, 0, alpha=0.4,
                              color=colors.get(col, "grey"), label=col)
        ax2.set_title("Drawdowns (%)", fontweight="bold")
        ax2.set_ylabel("Drawdown %")
        ax2.legend(fontsize=8)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha="right")

        # ── Panel 3: R² comparison bar ────────────────────────────────────
        ax3 = fig.add_subplot(gs[2, 0])
        models = reg_df["model"].tolist()
        r2s    = reg_df["R2"].tolist()
        bar_colors = ["#b71c1c", "#1565c0", "#6a1b9a"]
        bars = ax3.bar(models, r2s, color=bar_colors, alpha=0.8, width=0.5)
        ax3.bar_label(bars, fmt="%.4f", fontsize=8)
        ax3.set_title("Predictive R² (DV: next-day S&P return)",
                       fontweight="bold")
        ax3.set_ylabel("R²")
        ax3.set_ylim(0, max(r2s) * 1.4 + 0.001)
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=15, ha="right")

        # ── Panel 4: Sharpe + Hit rate comparison ─────────────────────────
        ax4 = fig.add_subplot(gs[2, 1])
        strats  = met_df["strategy"].tolist()
        sharpes = [float(s) for s in met_df["Sharpe"].tolist()]
        bar_c4  = [colors.get(s, "grey") for s in strats]
        bars4   = ax4.bar(strats, sharpes, color=bar_c4, alpha=0.8, width=0.5)
        ax4.bar_label(bars4, fmt="%.3f", fontsize=8)
        ax4.set_title("Annualised Sharpe Ratio by Strategy", fontweight="bold")
        ax4.set_ylabel("Sharpe")
        ax4.axhline(0, color="black", lw=0.7)
        plt.setp(ax4.xaxis.get_majorticklabels(), rotation=15, ha="right")

        fig.suptitle("Fear & Greed Index vs VIX – S&P 500 Performance Comparison",
                      fontsize=12, fontweight="bold", y=1.01)

        fig.savefig(f"{OUTPUTS_DIR}/performance/fg_vs_vix_summary.png",
                    bbox_inches="tight")
        plt.close(fig)
        print(f"[save] fg_vs_vix_summary.png")

    # ── FG index vs VIX scatter coloured by fwd return ────────────────────
    with plt.style.context(STYLE):
        fig2, axes = plt.subplots(1, 2, figsize=(13, 5))

        for ax, xcol, xlabel in [
            (axes[0], "vix_close",  "VIX"),
            (axes[1], "fg_primary", "Fear & Greed Index"),
        ]:
            sub = df[[xcol, "sp500_fwd_1d"]].dropna()
            sc  = ax.scatter(sub[xcol], sub["sp500_fwd_1d"] * 100,
                              c=sub["sp500_fwd_1d"] * 100,
                              cmap="RdYlGn", alpha=0.5, s=12,
                              vmin=-3, vmax=3)
            # Trend line
            coef = np.polyfit(sub[xcol], sub["sp500_fwd_1d"], 1)
            xs   = np.linspace(sub[xcol].min(), sub[xcol].max(), 200)
            ax.plot(xs, np.polyval(coef, xs) * 100,
                    color="black", lw=1.5, ls="--")
            r, p = stats.pearsonr(sub[xcol], sub["sp500_fwd_1d"])
            ax.set_title(f"{xlabel} vs Next-Day S&P Return\nr={r:.3f}  p={p:.3f}",
                          fontsize=9, fontweight="bold")
            ax.set_xlabel(xlabel)
            ax.set_ylabel("Fwd 1d Return (%)")
            plt.colorbar(sc, ax=ax, label="Return %")

        plt.tight_layout()
        fig2.savefig(f"{OUTPUTS_DIR}/performance/fg_vs_vix_scatter.png",
                     bbox_inches="tight")
        plt.close(fig2)
        print(f"[save] fg_vs_vix_scatter.png")


# ── Main ───────────────────────────────────────────────────────────────────

def main():
    df = load()

    reg_df          = regression_comparison(df)
    dir_df          = directional_accuracy(df)
    equity, met_df  = backtest(df)
    plot_summary(df, equity, reg_df, dir_df, met_df)

    print(f"\n[performance] All outputs in {OUTPUTS_DIR}/performance/")
    print("\n══ Final Summary ════════════════════════════════════════════════")
    print(met_df.to_string(index=False))


if __name__ == "__main__":
    main()

[load] 488 rows after join

══ A. Predictive Regressions (DV = sp500_fwd_1d) ═══════════════

  [VIX only]  R²=0.0179  adj-R²=0.0159  F-p=0.1034
    vix_z      coef=+0.00136  p=0.1028 

  [FG only]  R²=0.0017  adj-R²=-0.0004  F-p=0.4029
    fg_z       coef=+0.00041  p=0.4025 

  [VIX + FG]  R²=0.0233  adj-R²=0.0193  F-p=0.218
    vix_z      coef=+0.00208  p=0.0886 *
    fg_z       coef=-0.00103  p=0.2451 

══ B. Directional Accuracy ══════════════════════════════════════
  signal  hit_rate   n  chi2  p_chi2 significant
FG Index    0.5328 488 0.332  0.5643           ✗
     VIX    0.5328 488 0.756  0.3845           ✗

══ C. Backtest Results ══════════════════════════════════════════
  strategy total_ret  CAGR ann_vol Sharpe max_DD  days_long pct_long
Buy & Hold     45.5% 21.4%   16.1%  1.283 -18.9%        488   100.0%
 VIX (<20)     18.2%  9.0%   10.5%  0.880  -9.9%        399    81.8%
  FG (<55)     23.2% 11.4%   11.5%  0.996 -12.4%        386    79.1%
[save] fg_vs_vix_summary.png
[save